# CinePal — GPU Embedding on Colab

Runs the full `download → clean → split → embed → save artifacts → upload to HF` pipeline
on a free Colab GPU. Teammates then fetch the artifacts locally with:

```bash
python -m db.ingestion.fetch
python -m db.ingest --ingest all --from-artifact
```

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Click the 🔑 *Secrets* icon (left sidebar) and add:
   - `KAGGLE_USERNAME` / `KAGGLE_KEY` — from your Kaggle account settings → API
   - `HF_TOKEN` — from huggingface.co → Settings → Access Tokens (write scope)
   - `CINEPAL_ARTIFACTS_REPO` — e.g. `yourname/cinepal-embeddings`
   - `GITHUB_TOKEN` *(only if the repo is private)* — a PAT with `repo` scope
3. Fill in `REPO_URL` in cell 3.

In [ ]:
# 1 — Verify GPU is available
!nvidia-smi

In [ ]:
# 2 — Install project dependencies
!pip install -q sentence-transformers pyarrow pandas numpy huggingface_hub kaggle pyyaml python-dotenv tqdm psycopg[binary] pgvector annotated-types pydantic

In [ ]:
# 3 — Clone the repo and add it to the Python path
import os
import sys
from google.colab import userdata

REPO_URL = "https://github.com/YOUR_ORG/cantucci.git"  # <-- fill in before running

# For private repos, inject the GitHub token into the clone URL
github_token = userdata.get("GITHUB_TOKEN") if "GITHUB_TOKEN" in os.environ or True else None
try:
    github_token = userdata.get("GITHUB_TOKEN")
    if github_token:
        REPO_URL = REPO_URL.replace("https://", f"https://{github_token}@")
except Exception:
    pass  # secret not set — assume public repo

if not os.path.isdir("cantucci"):
    os.system(f"git clone {REPO_URL} cantucci")
else:
    os.system("git -C cantucci pull --ff-only")

os.chdir("cantucci")
if "." not in sys.path:
    sys.path.insert(0, ".")

print("Working directory:", os.getcwd())

In [ ]:
# 4 — Set up Kaggle credentials from Colab secrets
import json
from pathlib import Path

kaggle_dir = Path(os.path.expanduser("~/.kaggle"))
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"
kaggle_json.write_text(json.dumps({
    "username": userdata.get("KAGGLE_USERNAME"),
    "key": userdata.get("KAGGLE_KEY"),
}))
kaggle_json.chmod(0o600)
print("Kaggle credentials written.")

In [ ]:
# 5 — Download raw data, clean, and split
from db.ingestion import download, clean, split

RAW_DIR = Path("data/raw")
ARTIFACTS_DIR = Path("data/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

download.fetch(RAW_DIR)
df = clean.prepare(RAW_DIR)
main_df, mini_df, eval_df = split.three_way(df, mini_size=200, eval_frac=0.10, seed=42)

print(f"Splits — main: {len(main_df)}, mini: {len(mini_df)}, eval: {len(eval_df)}")

In [ ]:
# 6 — Embed on GPU
import torch
import numpy as np
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {device}")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

all_texts = (
    list(main_df["composite_text"])
    + list(mini_df["composite_text"])
    + list(eval_df["composite_text"])
)
print(f"Encoding {len(all_texts)} texts with batch_size=1024 ...")

all_emb = model.encode(
    all_texts,
    batch_size=1024,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

assert all_emb.shape == (len(all_texts), 384), f"Unexpected shape: {all_emb.shape}"
print(f"Done. Shape: {all_emb.shape}")

In [ ]:
# 7 — Save artifacts (reuses _save from db.ingest so the format stays in sync)
from db.ingest import _save

n_main, n_mini = len(main_df), len(mini_df)
main_emb = all_emb[:n_main]
mini_emb = all_emb[n_main : n_main + n_mini]
eval_emb = all_emb[n_main + n_mini :]

_save(main_df, main_emb, ARTIFACTS_DIR / "main.parquet")
_save(mini_df, mini_emb, ARTIFACTS_DIR / "mini.parquet")
_save(eval_df, eval_emb, ARTIFACTS_DIR / "eval_holdout.parquet")

for p in sorted(ARTIFACTS_DIR.glob("*.parquet")):
    print(f"{p.name}: {p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# 8 — Upload to Hugging Face Datasets
from huggingface_hub import HfApi

HF_REPO = userdata.get("CINEPAL_ARTIFACTS_REPO")  # e.g. "yourname/cinepal-embeddings"
HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi()
api.create_repo(repo_id=HF_REPO, repo_type="dataset", exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=str(ARTIFACTS_DIR),
    repo_id=HF_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
    commit_message="Update movie embeddings (all-MiniLM-L6-v2, seed=42)",
)
print(f"\nArtifacts published: https://huggingface.co/datasets/{HF_REPO}")

## Done

Teammates can now ingest without a GPU:

```bash
# .env must have CINEPAL_ARTIFACTS_REPO set (and HF_TOKEN if the repo is private)
python -m db.ingestion.fetch
python -m db.ingest --ingest all --from-artifact
```

> **Note on GPU vs CPU floating-point:** embedding outputs may differ by ~1e-6 between
> devices. This is below any meaningful threshold for cosine similarity and can be ignored.